AI use disclaimer

AI was used in certain ways during this assignment. They are as follows:
1. To generate suggestions on data imputation/scaling/encoding methods;
2. To explain various pieces of documentation;
3. For debugging purposes;
4. To generate examples on how to implement or use various unfamiliar libraries.

In [ ]:
%pip install kaggle
%pip install scikit-learn
%pip install python-dotenv 
# I was unable to download a kaggle.json file for some reason, so I used environment variables to download the Kaggle dataset instead.

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

# loading data 
pd.set_option('display.max_columns', None)
pd.option_context('display.max_rows', None)
banking_df = pd.read_csv("data/bank-additional.csv", sep = ";")

**Ordering of tasks**
1. Data Loading and Exploration -- this helps identify outliers, presence of skewed variables, separate categorical/numerical variables, missing variables, and class imbalance. 
2. Identifying the prediction target
3. Data Splitting: After the EDA step, the next step is to split the data into train, test and validation steps. This is because steps such as data imputation, handling class imbalance through methods such as SMOTE and ADASYN, could lead to data leakage if performed before splitting the dataset. (Eg, if missing values are imputed through mean/mode, imputation before splitting would reveal information about the entire dataset; if feature scaling is done before splitting, this would result in distributional statistics being computed on the full dataset and expose the model to the test set.)
4. Managing missing variables -- This is to be done strictly after splitting to prevent data leakage. Imputation has to be fitted strictly using the training set, as computing the mode from the entire dataset could lead to the model 'seeing' information from the holdout sets. 
5. Feature scaling --  Scaling must also be done on the training set only, strictly after the train test split. This is because scaling uses distributional statistics such as sample mean. By scaling before the split, the model will be exposed to information from the holdout sets, leading to data leakage. 
6. Feature selection -- Feature selection must also be done after the split to prevent data leakage. If the model is exposed to information in the test and validation sets, the model is exposed to patterns in the holdout sets and this will lead to overfitting. 
7. Training the logistic model.

**Task 1: Data Exploration**

In [ ]:
banking_df.head(50)

In [ ]:
print(banking_df.shape)
print(banking_df.info())

In [ ]:
banking_df.describe()

In [ ]:
categorical_columns = banking_df.select_dtypes(include=['object'])
for column in categorical_columns:
    print(banking_df[column].value_counts())

Information about the dataset 

There are 21 columns, which can be split into four categories:
1. Information about the bank client (eg age and job.)
2. Information regarding the last contact which has been made with the client (eg contact)
3. Other regarding the contacts made with the client during the current campaign (eg campaign)
4. socioeconomic factors, such as cons.price.idx

In addition, the target that we are trying to predict (whether or not the customer will subscribe to a term deposit) is the column "y" (yes or no). 

There are 4119 entries in total, 21 columns, and although there are no missing entries, there are implicit missing values, with missing information being labelled as unknown in various columns.

This information is supported by online sources which I managed to find about the UCI dataset. The links to these sources are given here: (https://archive.ics.uci.edu/dataset/222/bank+marketing, https://github.com/alexkataev/Case-Study-UCI-Bank-Marketing-Dataset)


**Task 2: Identifying the prediction target**
The target that we are trying to predict is the column "y", which is yes, or no. This indicates whether the customer will subscribe to the term deposit. 

Two variables that superficially appear to be the target are as follows:
1. poutcome: the name of this variable can be misleading if no further research is done as to the context of this problem. However, this should NOT be treated as the prediction outcome as it refers to success or failure in previous outreach attempts.
2. duration: duration represents the length of the last contact time with a specific client before the outcome of the campaign is known. Duration can be seen as a proxy for the target variable, as it is very strongly correlated with the outcome, "y". Typically, with calls that are very short in duration, it shows that customers lack interest in subscribing.Duration is also not available at prediction time. This is because if we know the length of the last contact with the customer before success or failure, then we already know the outcome. In addition, we know that this assignment is for training a logistic regression model, meaning our prediction target is binary. 


Information on each of the columns 
1. Age: numerical
2. Job: categorical (admin, blue color, unknown etc)
3. marital: categorical (married, single, divorced, unknown)
4. education: categorical (basic.4y, basic.6y etc)
5. default: whether the client has credit in default. categorical (no, yes, unknown)
6. housing: whether the client holds a housing loan, categorical (yes, no, unknown)
7. loan: whether the client has a personal loan, categorical (yes, no, unknown)
8. contact: the method used to reach the client, categorical (cellular, telephone)
9. month: the last contact month of the year. categorical (jan, feb etc)
10. day_of_week: the day of the week the customer was last contacted on: categorical (mon-fri)
11. poutcome: the outcome of previous campaigns in which the customer was reached out to, categorical. (failure, success, nonexistent-- meaning this is the first campaign in which customer is contacted)
12. previous: the number of times in previous marketing campaigns in which the customer was contacted, numerical. 
13. y: the target we are predicting (whether the customer will subscribe). (yes, no)
14. duration: the length of the last contact with the customer, in seconds. Numerical. 
15. campaign: Number of contacts performed during this campaign, numerical. 
16. pdays: number of days since the client was previously contacted, numerical. 
17. emp.var.rate: employment variation rate. Numeric
18. cons.price.idx: consumer price index. Numeric
19. cons.conf.idx: consumer confidence index. Numeric
20. euribor3m: euribor 3 month rate. Numeric. 
21: nr.employed: number of employees. Numeric

Next, I will plot a bar chart to visualise the distribution of the prediction target.

In [ ]:
sns.countplot(x = "y", data = banking_df)
plt.title("Marketing campaign success distribution")
absolute_labels = banking_df["y"].value_counts()
percentage_labels = banking_df["y"].value_counts() / len(banking_df) * 100
lbls = [f'{p[0]} ({p[1]:.0f}%)' for p in zip(absolute_labels, percentage_labels)]
plt.bar_label(container=plt.gca().containers[0], labels=lbls)
plt.xlabel("Subscribed to term deposit?")
plt.show()

As can be seen, the success count of 451 is only 11%. This is a clear case of data imbalance which should be dealt with later on in the data preprocessing process, to lower the risk of "no" being predicted simply because it is much more common of an outcome. 

I will do some basic visualizations for all the numerical and categorical features.

In [ ]:
banking_df.hist(figsize = (10,10), bins = 20)
plt.show()


Right now, it can be observed that the vast majority of rows have pdays as being 999. This raises some questions immediately, and it is likely that '999' for the pdays column is a special value, likely indicating that the individual has never been contacted before. I will confirm this by seeing whether for all the columns in which pdays is 999, poutcome is 'nonexistent'.

In [ ]:
sns.boxplot(x = "poutcome", y = "pdays", data=banking_df)
plt.title("Boxplot of pdays by poutcome")
plt.show()

The side by side boxplot seen above confirms my hypothesis. This is because while the values for pdays are "squashed" into a very small range when customers have been previously contacted before, all customers who have not been previously contacted would have a pdays value of 999. As the column 'poutcome' already includes the information of whether or not a customer has been previously conducted and the vast majority of entries in the pdays outcome is 999, pdays is uninformative and would likely immediately be dropped at the feature selection stage due to lack of variance. Due to the fact that we are dealing with a column with over 90% missing values that would definitely be removed at feature selection, I propose dropping the column "pdays".

In [ ]:
banking_df.drop(columns = ["pdays"], inplace = True)
banking_df.head()

In [ ]:
categorical_columns = banking_df.select_dtypes(include = ["object"])
for column in categorical_columns:
    if column == "y":
        continue
    sns.countplot(x = column, data = banking_df)
    plt.xticks(rotation = 45, fontsize = 8)
    plt.show()

It can be seen that a huge proportion of the values in the poutcome column are "nonexistent". However, we do not drop this column as the value 'nonexistent' is actually highly informative, representing customers who have never been contacted for previous campaigns before and how this may affect the target. Therefore, we will keep this and use one hot encoding to represent the three classes of values.

Next, I will examine the proportion of people who will subscribe to the term deposit based on categorical features such as job and marital status. This is because I have a hypothesis that those of certain employment/educational backgrounds are more likely to subscribe than others. 

In [ ]:
def yes_distribution_by_category(column):
    sns.catplot(x = column, hue = "y", kind = "count", data = banking_df, width = 0.6)
    plt.xticks(rotation = 45, fontsize = 8) # to make the labels easier to read
    plt.title(f"Distribution of marketing campaign success by {column} category")
    plt.show()
    print(pd.crosstab(banking_df[column], banking_df["y"], normalize = "index"))


for column in categorical_columns:
    if column == "y":
        continue
    yes_distribution_by_category(column)


Based on a visual inspection and the cross tabulation, it appears that certain job types such as admin, student, and retired have a higher proportion of successful attempts. In addition, it can be seen that the proportion of customers who subscribed to the term deposit is significantly higher for groups with higher educational levels, eg university degree.

In [ ]:
duration_with_yes = banking_df[banking_df["y"] == "yes"]["duration"]
duration_with_no = banking_df[banking_df["y"] == "no"]["duration"]
plt.boxplot([duration_with_yes, duration_with_no], labels = ["yes", "no"])
plt.title("Duration distribution with marketing campaign result")
plt.ylabel("Duration in seconds")
plt.show()

It can be seen that the the median and third quartile values for duration are significantly lower if the outcome is "no". 
After an analysis of the various features, it is observed that the feature "duration" should be excluded, as it leaks information about the prediction outcome. As stated earlier, duration is unavailable at prediction time and is a proxy for our target.
We will have to drop this column before the rest of the steps to prevent data leakage. 
We cannot wait until the feature selection step to drop this column, as it would likely be treated as the most important column during feature selection.

In [ ]:
banking_df.drop(columns = ["duration"], inplace = True)
banking_df.head()

As shown earlier, we are dealing with a case of class imbalance, as only 11% of outreach attempts lead to success. When splitting the dataset, we need stratification. 

**Task 3: Splitting dataset**

In [ ]:

from sklearn.model_selection import train_test_split 

Since we are working with only 4119 examples, the dataset is relatively small. For larger datasets, the proportion of examples used for the holdout sets can be smaller, such as a 98/1/1 split for very large datasets as even 1% of the data represents a substantial number of examples.(source: https://www.lightly.ai/blog/train-test-validation-split#:~:text=Common%20split%20ratios%20include%2070,20%25%20testing%2C%20are%20common.)
In this case, we will work with a 60/20/20 split, as it is a very common three way split ratio.

In [ ]:
x = banking_df.drop("y", axis = 1)
y = banking_df["y"]
X_train_full, X_test, y_train_full, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y) #separate out the testing set first
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size = 0.25, random_state = 42, stratify = y_train_full) #separate out the validation set 
for name, labels in [("Train", y_train), ("Test", y_test), ("Validation", y_val)]:
    print(f"{name} class distribution")
    print(labels.value_counts(normalize = True))
# the class distribution is retained (89% no, 11% yes). 

**Task 4: Managing missing values**

1. Job, marital status, housing and loan: The proportion of missing values is very low. This likely means that the missingness is non informative, and could be the result of failures during data entry. I will choose to impute these categories with the mode.
2. Education and default: The proportion of missing values for these two columns are higher than the proportion of missing values for job and marital status. In addition, I believe that unknown values in these two categories are likely to be informative. For instance, there are close to 0 entries whereby education level was listed as 'illiterate', while over 100 entries are listed as 'unknown'. It could be the case that individuals who are illiterate or did not complete any formal education simply refused to admit their actual education level. The same logic could be applied to the default column, as those who have previously defaulted on their loans may have refused to admit that information. I will treat the unknown values as a separate category during the encoding process.
3. poutcome: The proportion of implicit missing values "unknown" is very high. However, these missing values are informative, and should be treated as a separate category instead, as the missing values encode important information on whether the customer has been contacted in previous campaigns or not. 

Next, I will determine the types of categorical variables I am working with, as well as how to encode each categorical variable. 

1. Job: Nominal. There is no order between categories of jobs. 
2. Marital: Nominal. 
3. Education: Ordinal. There is a clear progression in education levels. 
4. Default: nominal. 
5. Housing and loan: Binary. As stated earlier, we will be imputing unknown values as the mode for these two columns, and thus there will only be two categories for these columns. 
6. contact: binary.
7. month, day_of_week: Cyclical. This is because Sunday loops back to Monday, December loops back to January and so on. I will consider using a cyclical encoder for converting these features into numeric columns, as this will help improve the predictive power of the model.
8. poutcome: nominal. 



As shown earlier, the data is highly imbalanced. This leads to poor performance on the minority class (in this case, the label "yes".) This was actually shown in an earlier version of my project, where the accuracy rate of my initial model was 90% but the recall rate (using the prediction of "yes" as a positive case) was only 12%, showing that the model is too liberal with predicting the majority class. To deal with this data imbalance, I attempted using the oversampling technique SMOTE. 
As SMOTE generates synthetic examples for the minority class through picking the k nearest neighbours of a random example of the minority class, it would be important to only use SMOTE AFTER the train/test/split and generate synthetic examples on the training class only, as not doing so will result in data leakage (examples from the holdout sets will be exposed.) SMOTE also needs to be carried out AFTER preprocessing, as it generates synthetic samples by calculating the distance between examples in a sample space.
However, after attempting to run SMOTE, I found that it was too computationally expensive and prevented my code from running. This is because it relies on K nearest neighbours computations. 
I also considered setting class_weight = balanced as a parameter to the Logistic Regression classifier. After testing out changing this parameter as well as RandomOverSampler, I found that the two methods yielded comparable results. 
Therefore, I decided on RandomOverSampler as my method of dealing with class imbalance.

In [ ]:
%pip install -q --upgrade imbalanced-learn

In [ ]:
# importing necessary libraries for data preprocessing and modeling 
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder,  QuantileTransformer , FunctionTransformer #use this to create sine cosine encoding 
from sklearn.impute import SimpleImputer # replace missing values with descriptive statistic 
from sklearn.compose import ColumnTransformer # column transformer can bundle together different preprocessing steps
from sklearn.linear_model import LogisticRegression 
from imblearn.pipeline import Pipeline  


The first step is to generate groups of features. This is because different features will undergo different steps for data preprocessing and scaling. For instance, columns which have missing values will need to undergo imputation. Categorical variables will additionally have to be encoded into numerical values. The types of scaling method that will be used will also be different, depending on whether data is nominal/ordinal/cyclical for categorical features, as well as the scale and distribution of data.

In [ ]:
# numerical columns 
numerical_columns = ["age", "campaign", "previous", "emp.var.rate", "cons.price.idx", 
                     "cons.conf.idx", "euribor3m", "nr.employed"]
# defining the sub pipeline used for the numerical columns. 
numerical_pipeline = Pipeline([("scaler", QuantileTransformer(output_distribution="normal"))])

In this case, we will use QuantileTransformer to scale the numerical columns. This is because many of the numeric columns are highly skewed or multimodal, which can affect model performance. 

There are many sources which state that machine learning algorithms tend to perform better on normally distributed variables. (https://machinelearningmastery.com/quantile-transforms-for-machine-learning/) QuantileTransformer is able to reduce the impact of outliers on data, and is therefore better able to handle skewness and multimodal data. (https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.QuantileTransformer.html)

In [ ]:
nominal_mode_impute_columns = ["job", "marital", "housing", "loan"] # these columns have missing values, but the proportion of missing values is small. 

nominal_mode_impute_pipeline = Pipeline([("imputer", SimpleImputer(strategy = "most_frequent")), 
                                         ("encoder", OneHotEncoder(handle_unknown = "ignore", sparse_output = False))])

These are nominal features which, as shown earlier, have a lower proportion of unknown values that are likely to be uninformative, eg errors caused by mistakes during data handling. Therefore, we will choose to impute the missing values with the mode values, followed by one hot encoding to convert the categorical feature into numerical values.

In [ ]:
nominal_keep_columns = ["default", "contact", "poutcome"] # do not impute missing values for these columns, only encode. 
nominal_keep_pipeline = Pipeline([("encoder", OneHotEncoder(handle_unknown = "ignore", sparse_output = False))])

For contact, as there are no missing values, it would be unnecessary to conduct imputation. 
For default and poutcome columns, as mentioned earlier, the high proportion of unknown values are likely to be informative and they will be kept as a separate category during one hot encoding. 

In [ ]:
education_categories = ["illiterate", "basic.4y", "basic.6y", "basic.9y", "high.school", "professional.course", "university.degree", "unknown"]
ordinal_columns = ["education"]
ordinal_pipeline = Pipeline([("encoder", OrdinalEncoder(categories = [education_categories], handle_unknown = "use_encoded_value", unknown_value = -1))])

In this case, I use OrdinalEncoder for education as education levels follow a progression. For instance, university degrees mean more years of formal education than a person who underwent high school only. As the "unknown" category is hypothesised to be informative, I will encode "unknown" as a separate category.

In [ ]:
cyclical_columns = ["month", "day_of_week"] # march to december, and monday to friday.

# first step is to replace the month and day_of_week with corresponding numerical values
month_map = {"mar": 3, "apr": 4, "may": 5, "jun": 6, "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}
day_map = {"mon": 1, "tue": 2, "wed": 3, "thu": 4, "fri": 5}

dataframes_to_map = [X_train, X_val, X_test]

for df in dataframes_to_map:
    df["month"] = df["month"].map(month_map)
    df["day_of_week"] = df["day_of_week"].map(day_map)
    print(f"Dataframe after mapping:")
    print(df.head())


In [ ]:
import numpy as np

def encode_sin_cos(columns):
    columns = np.array(columns) # have to convert to np array, if not a pd dataframe is passed
    print(f"Datatype of input columns: {columns.dtype}")
    month = columns[:, 0] # selects all rows with the first column, which is month
    day_of_week = columns[:, 1]
    month_sin = np.sin((2 * np.pi * month) / 12)  # maximum value of month is 12
    month_cos = np.cos((2 * np.pi * month) / 12)
    day_of_week_sin = np.sin((2 * np.pi * day_of_week) / 5) # maximum value of day is 5, only goes to friday
    day_of_week_cos = np.cos((2 * np.pi * day_of_week) / 5)
    return np.column_stack((month_sin, month_cos, day_of_week_sin, day_of_week_cos))


cyclical_encoder = FunctionTransformer(func= encode_sin_cos)

cyclical_pipeline = Pipeline([("encoder", cyclical_encoder)])



In [ ]:
preprocessor = ColumnTransformer(transformers = [("num", numerical_pipeline, numerical_columns), ("nominal_mode_impute", nominal_mode_impute_pipeline, nominal_mode_impute_columns), ("nominal_keep", nominal_keep_pipeline, nominal_keep_columns),
                                                ("ordinal", ordinal_pipeline, ordinal_columns), ("cyclical", cyclical_pipeline, cyclical_columns)])

In [ ]:

from imblearn.over_sampling import RandomOverSampler
oversampler = RandomOverSampler(random_state = 42)

**Task 5: Feature selection**

Next, I will consider various feature selection methods. I considered using SelectKBest, but this turned out not to be a good choice as SelectKBest selects features using univariate statistics -- it does not consider whether variables are multicollinear, only scoring features based on their relationship with the target variable. As logistic regression requires input variables to be non multicollinear, SelectKBest is inappropriate. I then considered using Boruta or L1 regularisation for feature selection. As Boruta tends to be slower (runs on a random forest algorithm) and is less simple to integrate  into the pipeline, I decided to try L1 regularisation. L1 Regularisation has the effect of implicit feature selection as it penalises model complexity, pushing the coefficients of redundant or noisy features to zero. 
Features that are highly correlated with each other or have a huge proportion of missing values are likely to be removed during feature selection.

In [ ]:
pipe = Pipeline([("preprocessor", preprocessor), 
                 ("oversampler", oversampler), 
                 ("model", LogisticRegression(l1_ratio=1, solver = "liblinear", max_iter= 1000, C= 0.1)) ])
# the optimiser is set to liblinear because it supports lasso regularisation


**Task 6: Model Training**

The next step is to consider model training. 
Firstly, we must define some performance metrics. In this case, the distributions of labels is very imbalanced, as 89% of reach-out attempts end up in failure. This would make accuracy alone a dubious statistic, as a model can predict failure in every case and still end up with a very high accuracy. Therefore, we will also use Cohen's Kappa statistic, as it applies to imbalanced problems and measures how much better a model performs compared to a classifier that randomly makes predictions based on the frequencies of classes.
1. Precision, which shows the false positive rate, 
2. Recall, which shows the false negative rate, 
3. Accuracy, which is the total correct predictions rate. 
4. Cohen's Kappa Statistic

Secondly, we must define the baseline that we will compare our model with. The baseline chosen is a zero-rule baseline algorithm, which will predict the most common label in the training set. 


In [ ]:
# defining a helper function to generate the confusion matrix and other evaluation metrics.
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, cohen_kappa_score

def score_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    print("Label     :",list(y_test))
    print("Prediction:",list(y_pred))
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {round(accuracy*100, 2)}%")
    model_confusion_matrix = confusion_matrix(y_test, y_pred, labels = model.classes_)
    disp = ConfusionMatrixDisplay(confusion_matrix = model_confusion_matrix, display_labels = model.classes_)
    disp.plot()
    plt.title("Confusion matrix")
    plt.show()
    precision = precision_score(y_test, y_pred, pos_label = "yes")
    print(f"Precision: {round(precision*100, 2)}%") 
    recall = recall_score(y_test, y_pred, pos_label = "yes")
    print(f"Recall: {round(recall*100, 2)}%")
    return y_pred


In [ ]:
from sklearn.dummy import DummyClassifier 
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, cohen_kappa_score

dummy_clf = DummyClassifier(strategy = "most_frequent")
dummy_clf.fit(X_train, y_train)
score_model(dummy_clf, X_test, y_test)
# there is no need to calculate cohen Kappa score for the dummy classifier. It will be 0 as the model is making random predictions. Any agreement between actual and predicted values is by chance only.



It can be seen that the accuracy score of the dummy classifier is very high (89%). On the other hand, recall and precision are 0 as the classifier never predicts a successful outcome. This is a good example of how accuracy is a flawed metric for the task at hand. This is because while oversampling was applied to the training set, the class distributions in the test set remained unaffected, and the dummy classifier simply predicting "no" in every case would be correct in 89% of the case.

In [ ]:
pipe.fit(X_train, y_train)
predicted_values = score_model(pipe, X_test, y_test)
cohen_kappa = cohen_kappa_score(y_test, predicted_values)
print(f"Cohen's kappa statistic for logistic regression: {round(cohen_kappa*100, 2)}%")

The accuracy rate for logistic regression is lower than the zero-rule baseline model. This is because the model is more conservative with predicting the majority class "no". In addition, the precision rate for predicting the minority class is higher (albeit still low) at 28.8%. Recall is significantly higher at 54.4%, as the number of missed potential subscribers of the term deposit is drastically lowered.
In the context of predicting whether someone would subscribe to a bank deposit, missed potential subscribers is arguably more costly to the bank than the time, effort, and cost spent needlessly contacting people who would end up not subscribing. Therefore, I could argue that recall is a more important metric in this case than either accuracy or precision, and this logistic regression model is a improvement over the baseline. 
This conclusion is further supported by the fact that the Cohen's Kappa value is 27%, which is considered a fair improvement over random chance. (https://numiqo.com/tutorial/cohens-kappa)

**Model improvements**

Although hyperparameter tuning is outside the scope of this assignment, I wished to test it out and see if it would improve the model performance. 
As mentioned above, I decided to use L1 regularisation for feature selection. The parameter, c, controls the regularisation strength. The smaller the c value, the greater the regularistion strength and the lower the variance. As can be seen, the c value chosen was 0.1. I have a hypothesis that the low c value resulted in underfitting of the training data. 
I will test out GridSearchCV, setting the scoring value as recall. This is because as stated earlier, missed potential subscribers represent a huge cost for the bank, and recall is the most important metric that will be used to select the best value of c.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer 
recall_scorer = make_scorer(recall_score, pos_label = "yes")


param_grid = dict(model__C = [0.01, 0.1, 1, 10, 100])
grid_search = GridSearchCV(pipe, param_grid = param_grid, scoring = recall_scorer)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_
print(f"Best parameters: {best_params}")
best_model = grid_search.best_estimator_
y_val_pred = best_model.predict(X_val)
best_model_accuracy = accuracy_score(y_val, y_val_pred)
best_model_precsion = precision_score(y_val, y_val_pred, pos_label = "yes")
best_model_recall = recall_score(y_val, y_val_pred, pos_label = "yes")
best_model_cohen_kappa = cohen_kappa_score(y_val, y_val_pred)
print(f"Accuracy score of best model: {round(best_model_accuracy * 100, 2)}%")
print(f"Precision score of best model: {round(best_model_precsion * 100, 2)}%")
print(f"Recall score of best model: {round(best_model_recall * 100, 2)}%")
print(f"Cohen's kappa statistic for best model: {round(best_model_cohen_kappa * 100, 2)}%")






While the recall of the model is significantly improved, the accuracy score, precision score, and Cohen's Kappa statistic for the model have decreased. This is because the model was tuned to minimise false negatives. The decrease in precision is a natural result of the precision-recall tradeoff. 
Overall, the conclusion of whether the model has improved after hyperparameter tuning would depend on the availability of context-specific data to determine which metric is the most important for the problem.. For instance, the bank could quantify the cost of losing a potential customer, vs the cost of unnecessarily contacting those unlikely to subscribe. 